# Chapter 2: B-Tree Basics
*From: Database Internals*

## TL;DR

This chapter answers one question: **what data structure should index records that live on disk?** The classical in-memory answer (balanced BSTs) fails; the answer the industry converged on is the B-Tree and its B+ variant.

- **Balanced BSTs are fine in RAM** (`O(log2 N)` lookups) but collapse on disk: binary fanout means `log2 N` seeks per query, random node placement destroys locality, and every insertion risks rebalancing and pointer updates.
- **Disk forces a block-I/O model.** HDDs pay tens of milliseconds per seek and reward sequential I/O; SSDs read/write whole pages and erase whole blocks via an FTL. Either way, one "access" transfers a block, not a byte.
- **Good on-disk structures maximise fanout and minimise height.** High fanout packs many separator keys into one page; low height means a root-to-leaf lookup crosses only a handful of blocks.
- **B+ Trees realise this.** Internal nodes hold only separator keys and child pointers; values live only in leaves; leaves link sibling-to-sibling for cheap range scans. Lookups cost `O(log_K M)` page transfers while still only `O(log2 M)` key comparisons.
- **Splits and merges maintain balance bottom-up.** Overflowing a node splits it in two and promotes a separator to the parent; underflowing nodes merge and demote a separator. Both can cascade to the root — and only root split/merge changes tree height.
- **Numbers to remember:** with fanout 500, a single root-to-leaf traversal indexes 4 billion keys in about 4 levels. A BST would need 32.

## Chapter Roadmap

| # | Concept | One-line summary |
|---|---------|------------------|
| 1 | Why BSTs fail on disk | Low fanout + pointer chasing + rebalancing kills on-disk BSTs. |
| 2 | Disk hardware & block I/O | HDD seeks and SSD page/erase-block semantics force block-level thinking. |
| 3 | On-disk structure design principles | High fanout, low height, locality, minimal out-of-page pointers. |
| 4 | B-Tree hierarchy & separator keys | Root / internal / leaf nodes; B+ variant pushes values to leaves. |
| 5 | Lookup algorithm & complexity | Single root-to-leaf descent: `log_K M` transfers, `log2 M` comparisons. |
| 6 | Node splits & merges | Bottom-up balancing; height changes only at root. |

---

## 1. Why Binary Search Trees Fail on Disk

A **binary search tree** stores a key and value at each node plus pointers to a left and right child. A node's key is greater than every key in its left subtree and less than every key in its right subtree. A search walks from the root, comparing and descending, until it finds the key or falls off a leaf.

In a **balanced** BST, every operation is `O(log2 N)` because each step halves the remaining search space. The balance invariant — heights of sibling subtrees differ by at most a small constant — has to be actively maintained. Inserts and deletes run a **rotation** step around a pivot node whenever they leave the tree skewed. Without balancing, inserts arriving in sorted order degenerate the tree into a linked list and lookups collapse to `O(N)`.

That pathological case is what makes a naive on-disk BST hopeless. But even the happy path is bad. BSTs have **fanout 2**, so a tree over `N` keys has height `log2 N`. Four billion keys = 32 levels. Each pointer hop between levels can land on a different page of the disk because nodes are allocated in insertion order, not in tree order. So a lookup needs ~32 random disk seeks. Balancing rotations relocate nodes and force pointer updates on neighbouring nodes, amplifying the pain. 2-3-Trees, AVL trees, and other low-fanout in-memory structures share the same ceiling.

```mermaid
graph TB
    subgraph Balanced[Balanced BST: O log N]
        B1[8]
        B1 --> B2[4]
        B1 --> B3[12]
        B2 --> B4[2]
        B2 --> B5[6]
        B3 --> B6[10]
        B3 --> B7[14]
    end

    subgraph Path[Pathological BST: O N]
        P1[2] --> P2[4]
        P2 --> P3[6]
        P3 --> P4[8]
        P4 --> P5[10]
        P5 --> P6[12]
        P6 --> P7[14]
    end
```

```mermaid
graph LR
    subgraph Before[Before rotation: left-heavy branch]
        A1[5] --> A2[3]
        A2 --> A3[1]
    end

    subgraph After[After rotating around 3]
        C1[3] --> C2[1]
        C1 --> C3[5]
    end
```

### Why BSTs are wrong for disk

| Problem | Consequence on disk |
|---------|---------------------|
| Fanout = 2 | Height = `log2 N` -> ~32 seeks for 4 billion keys |
| Nodes allocated in insertion order | No guarantee a child sits in the same block as its parent (poor locality) |
| Balancing = rotations | Relocates nodes and updates pointers on every insert/delete |
| One key per node | Huge pointer-to-payload overhead per page |
| Worst-case `O(N)` on skewed input | Even one bad workload period destroys performance permanently |

> **Takeaway:** BSTs optimise for cheap pointer-following. On disk, following a pointer may cost a seek. The whole premise of the data structure is wrong.

---

## 2. Disk Hardware and the Block I/O Model

Every on-disk data structure inherits constraints from the medium. There are two storage families to reason about.

### Hard Disk Drives

An HDD has a **spinning platter** and a **read/write head** on an arm. Reading a byte means rotating the platter until the byte is under the head, and moving the arm radially until the head is on the right track. That **seek** dominates latency (roughly 10 ms). Once positioned, reading contiguous bytes is cheap because the head is already where it needs to be. The smallest unit the drive will transfer is a **sector** (512 B to 4 KB). This is why HDD-era algorithms lean hard on sequential I/O: a random-vs-sequential latency ratio of 1000:1 is not unusual.

### Solid State Drives

An SSD has **no moving parts**. Instead it's built from memory cells, grouped into strings, strings into arrays, arrays into pages, pages into blocks, blocks into planes, planes onto dies. The key asymmetries:

- The smallest **read/program** (write) unit is a **page** (2-16 KB).
- The smallest **erase** unit is a **block** (64-512 pages) — the "erase block."
- You can't overwrite a page in place. You can only write to an erased cell; rewriting means erasing the whole surrounding block first.
- A **Flash Translation Layer (FTL)** maps logical page IDs to physical locations, tracks empty/written/discarded pages, and runs **garbage collection** that relocates still-live pages out of a block before erasing it.

SSDs have much smaller random-vs-sequential gap than HDDs (prefetching and internal parallelism still favour sequential), but unaligned random writes still cost extra because they trigger more GC work.

### The universal constraint: block devices

Whether HDD or SSD, the OS presents the disk as a **block device**. Reading a "single word" actually reads the whole block. Writing a byte means reading, modifying, and writing the whole block. You cannot opt out of this — you can only design around it.

```mermaid
graph TB
    Cell[Cell<br/>1 or more bits] --> String[String<br/>32-64 cells]
    String --> Array[Array<br/>many strings]
    Array --> Page[Page<br/>2-16 KB<br/>smallest read/write]
    Page --> Block[Block<br/>64-512 pages<br/>smallest erase]
    Block --> Plane[Plane]
    Plane --> Die[Die<br/>1+ per SSD]
    FTL[Flash Translation Layer<br/>page ID -> physical, GC] -. manages .-> Page
```

### HDD vs SSD at a glance

| Dimension | HDD | SSD |
|-----------|-----|-----|
| Moving parts | Yes (platter + head) | No |
| Dominant latency | Seek time (~10 ms) | Page read/program (~0.1 ms) |
| Smallest transfer unit | Sector (512 B - 4 KB) | Page (2-16 KB) |
| Smallest erase/rewrite unit | Sector (in place) | Block (64-512 pages) |
| Random vs sequential penalty | ~1000x | Small but non-zero (prefetch, parallelism, GC) |
| Write semantics | Overwrite in place | Erase-block, FTL remap, GC relocates live pages |
| Consequence for data structures | Favour long sequential runs | Favour full-block writes, avoid unaligned random writes |

> **Block-device takeaway:** you always transfer a whole block. A structure that reads one key per block is wasting that block's other 99% of bytes. Pack more usefully.

---

## 3. On-Disk Structure Design Principles

The block-I/O constraint plus the BST autopsy give us a short list of design rules that any serious on-disk index has to follow.

- **High fanout.** Every block you load should contribute many decisions to the search, not one. Pack dozens or hundreds of keys per node.
- **Low height.** Height = number of block transfers per lookup. Fanout and height are inversely related — high fanout buys low height automatically.
- **Locality.** Keep keys that are close in sort order physically close on disk, so range scans stream rather than seek.
- **Minimal out-of-page pointers.** Following a pointer that escapes the current page may cost a seek. Keep pointers short and local; compute offsets instead of chasing deep pointer chains.
- **Amortise balancing.** With high fanout, the frequency of structural changes drops naturally — splits and merges happen per-page, not per-key.
- **Accept internal fragmentation.** Reserving some empty space per node absorbs future insertions without immediate splits; 50-70% utilisation is a normal trade.

A **paged binary tree** (grouping BST nodes into pages) improves locality but still inherits fanout-2 tree height, pointer-heavy nodes, and balancing churn. It's a local fix, not the right abstraction.

```mermaid
graph LR
    subgraph LowF[Fanout = 2 BST over N = 1B keys]
        H1[Height = log2 1B = 30]
        H1 --> H2[~30 block transfers per lookup]
    end

    subgraph HighF[Fanout = 500 B-Tree over N = 1B keys]
        H3[Height = log500 1B = 4]
        H3 --> H4[~4 block transfers per lookup]
    end
```

### The trade-off cube for on-disk indexes

| Axis | Cheap side | Expensive side | What you're buying |
|------|------------|----------------|---------------------|
| Fanout | Higher = fewer levels | Larger nodes, more intra-node comparison | Fewer block transfers per lookup |
| Height | Lower | - | Fewer disk seeks |
| Node occupancy | Fuller = more keys per block | Risk of split on every insert | I/O efficiency vs write amplification |
| Pointer span | Intra-page | Cross-page | Locality vs flexibility |
| Balancing frequency | Rare (high fanout) | Every insert (low fanout) | Stable write latency |

---

## 4. B-Tree Hierarchy and Separator Keys

A **B-Tree** is a page-oriented balanced tree in which every node holds up to `N` keys and `N+1` child pointers. Nodes play one of three roles:

- **Root node** — the top of the tree; has no parent. There is exactly one.
- **Internal nodes** — everything between root and leaves. They navigate; they don't terminate searches.
- **Leaf nodes** — the bottom layer; no children.

Because B-Trees are **page organisation structures**, "node" and "page" are interchangeable. A node's **fanout** is the number of keys it currently holds; its **occupancy** is the ratio of used to available slots.

### The B+ Tree variant (the one everyone actually uses)

The textbook "B-Tree" lets values live at any level. The near-universal variant — **B+ Tree** — keeps values **only in leaf nodes**. Internal nodes hold only **separator keys** plus child pointers. This matters:

- Internal nodes become smaller, so fanout grows (more separators per page), so the tree gets shorter.
- All data operations (insert, update, delete, scan) mutate only leaves. Internal nodes change only when splits/merges propagate up.
- Leaves often carry **sibling pointers** (forward, and sometimes backward), turning range scans into a linear walk along the leaf level rather than an up-and-down tree traversal.

Most literature and most real systems (MySQL InnoDB, PostgreSQL btree indexes) use the term "B-Tree" but implement B+ Trees. We follow the same convention from here on.

### Separator-key invariants

For a node with separator keys `K1 < K2 < ... < Kn` and child pointers `P0, P1, ..., Pn`:

- `P0` points to the subtree whose keys are **strictly less than** `K1`.
- `Pn` points to the subtree whose keys are **greater than or equal to** `Kn`.
- For each `i` in between, `Pi` points to the subtree whose keys `Ks` satisfy `K_{i-1} <= Ks < K_i`.

```mermaid
graph TB
    Root["Root<br/>separator keys: 30 | 60"]
    Root --> I1["Internal<br/>10 | 20"]
    Root --> I2["Internal<br/>40 | 50"]
    Root --> I3["Internal<br/>70 | 85"]

    I1 --> L1["Leaf<br/>1 5 8"]
    I1 --> L2["Leaf<br/>11 15 18"]
    I1 --> L3["Leaf<br/>21 25 28"]

    I2 --> L4["Leaf<br/>31 36 39"]
    I2 --> L5["Leaf<br/>41 45 48"]
    I2 --> L6["Leaf<br/>51 55 58"]

    I3 --> L7["Leaf<br/>61 65 69"]
    I3 --> L8["Leaf<br/>71 78 82"]
    I3 --> L9["Leaf<br/>86 92 99"]

    L1 -. sibling .-> L2
    L2 -. sibling .-> L3
    L3 -. sibling .-> L4
    L4 -. sibling .-> L5
    L5 -. sibling .-> L6
    L6 -. sibling .-> L7
    L7 -. sibling .-> L8
    L8 -. sibling .-> L9
```

### B-Tree vs B+ Tree

| Aspect | B-Tree (classic) | B+ Tree (the common variant) |
|--------|------------------|-------------------------------|
| Where values live | Any node (root, internal, leaf) | Leaves only |
| Internal node size | Larger (keys + values + pointers) | Smaller (separators + pointers) -> higher fanout |
| Leaf sibling pointers | Typically none | Usually present, powering fast range scans |
| Update propagation | Any level can be touched directly | Only leaves change; upper levels change only during split/merge |
| Range scan | Tree traversal (up and down) | Linear walk along leaf chain |
| Real-world usage | Rare standalone; academic baseline | Default in InnoDB, PostgreSQL, LMDB, WiredTiger, etc. |

> **Bottom-up growth:** unlike BSTs, B-Trees grow *from the leaves up*. A new leaf appears because an old one split; a new internal level appears only when the root itself splits. This is what keeps them balanced by construction.

---

## 5. B-Tree Lookup Algorithm and Complexity

A lookup is a single **root-to-leaf** traversal. At each node, binary-search the keys for the **first separator greater than** the target, and follow the corresponding child pointer. Descend until you hit a leaf; there, either find an exact match (point query), or find the predecessor (useful for range scans and for inserts that need to know where the key *would* go).

```mermaid
graph TB
    R["Root: 30 | 60<br/>search for 45"]
    R -->|45 >= 30 and 45 < 60| I["Internal: 40 | 50"]
    I -->|45 >= 40 and 45 < 50| L["Leaf: 41 45 48"]
    L -->|binary search| Hit["found 45"]
```

### Two views of lookup cost

B-Tree lookup complexity is usually quoted as `O(log M)`, but the *base* of that log depends on which resource you're counting.

| Cost view | What you're counting | Logarithm base | Why |
|-----------|----------------------|----------------|-----|
| Page transfers (disk seeks) | Levels traversed (tree height `h`) | `K` = keys per node | Each level down divides the key space by the node's fanout |
| Key comparisons (CPU work) | Total keys examined across all nodes | `2` | Binary search inside each node halves the in-node search space |

So for `M` keys in a tree with fanout `K`:
- **Disk seeks per lookup** ~ `log_K M` (this is the height).
- **Comparisons per lookup** ~ `log2 M` (independent of fanout, to a constant factor).

With fanout 500 and `M` = 4 billion, the height is `log500 4e9` ≈ 4. You get the whole key out in **about four block transfers** while still making the same ~32 comparisons a balanced BST would — but 32 cache-friendly comparisons on RAM-resident pages, not 32 seeks.

### Point query vs range query

- **Point query:** descend to a leaf, confirm or refute the key.
- **Range query:** descend to the leaf containing the start of the range, then **walk sibling pointers** along the leaf level until the end predicate is satisfied. No further tree traversal needed.

---

## 6. B-Tree Node Splits and Merges

Structural changes happen only when nodes overflow (too full to insert) or underflow (too empty to justify their existence). Both propagate **upward** and can cascade to the root.

### Insert -> possible split

Insert finds the target leaf via the lookup algorithm and appends the key-value pair. A **split** is needed when:

- **Leaf node:** capacity `N` key-value pairs is already full; adding one more pushes it over.
- **Non-leaf node:** capacity `N+1` pointers is already full; a child split wants to add one more.

Split procedure:

1. Allocate a new sibling node.
2. Move half the elements (from the **split point** / midpoint onward) into the new sibling.
3. Place the incoming element into whichever half it belongs in (compare to the promoted key).
4. **Promote** the split-point key up to the parent, paired with a pointer to the new sibling.

If the parent is also full, it splits the same way. If the root splits, a **new root** is allocated holding only the promoted key and two pointers — and the tree grows one level taller. This is the *only* way B-Tree height increases.

```mermaid
graph TB
    subgraph BeforeSplit[Before: leaf is full N=4, inserting 11]
        PB["Parent: 5 | 15"]
        PB --> LBefore["Leaf: 7 9 13 14<br/>+ incoming 11"]
    end

    subgraph AfterSplit[After: split at midpoint, promote 13]
        PA["Parent: 5 | 13 | 15"]
        PA --> LA1["Leaf: 7 9 11"]
        PA --> LA2["Leaf: 13 14"]
        LA1 -. sibling .-> LA2
    end
```

### Delete -> possible merge

Delete finds the leaf, removes the entry. If the node's occupancy falls below a threshold, it **underflows** and one of two repairs happens:

- **Merge:** if the underflowing node and a sibling together fit in one node, concatenate them.
- **Rebalance:** if they don't fit together, **redistribute** keys between them to restore both above the threshold.

Merge is needed when:

- **Leaf nodes:** combined key-value count of two neighbouring nodes `<= N`.
- **Non-leaf nodes:** combined pointer count of two neighbours `<= N+1`.

Merge procedure:

1. Copy all elements from the right sibling into the left.
2. Remove the right sibling's pointer from the parent. For non-leaf merges, **demote** the parent's separator key into the merged node — because there is now one subtree instead of two, the separator between them is no longer needed at the parent level.
3. Free the right sibling page.

The parent now has one fewer pointer; if that causes it to underflow too, the merge cascades. Only when **two children of the root** merge into one — leaving the root with a single child — can the root be eliminated and the tree shrink by one level. Otherwise, height stays constant through deletion.

```mermaid
graph TB
    subgraph BeforeMerge[Before: two underflowing leaves share a parent]
        PBM["Parent: 10 | 20 | 30"]
        PBM --> LBM1["Leaf: 11 13"]
        PBM --> LBM2["Leaf: 21 22"]
        LBM1 -. sibling .-> LBM2
    end

    subgraph AfterMerge[After: deleted 22, combined fits in one leaf]
        PAM["Parent: 10 | 30"]
        PAM --> LAM["Leaf: 11 13 21"]
    end
```

```mermaid
graph TB
    subgraph NLBefore[Non-leaf merge, before: deleting 10 from left child]
        NLP1["Parent: 15"]
        NLP1 --> NLL1["Internal: 5 | 8"]
        NLP1 --> NLR1["Internal: 20 | 25"]
    end

    subgraph NLAfter[After: separator 15 demoted into merged node]
        NLP2["Parent: (one child)"]
        NLP2 --> NLM["Internal: 5 | 8 | 15 | 20 | 25"]
    end
```

### Split/merge summary

| Operation | Trigger | Core action | Separator key | Height effect |
|-----------|---------|-------------|---------------|----------------|
| Leaf split | Leaf overflow on insert | Allocate sibling, move half, choose target half for new entry | Promote midpoint up | Height grows only if root splits |
| Non-leaf split | Child split added a pointer beyond capacity | Same as leaf split, plus one extra pointer | Promote midpoint up | Height grows only if root splits |
| Leaf merge | Two siblings each underflow and fit together | Concatenate right into left | Remove separator from parent | Height shrinks only if root loses its last separator |
| Non-leaf merge | Same, but at internal level | Concatenate right into left | **Demote** parent's separator into merged node | Same |
| Rebalance | Underflow but combined doesn't fit | Redistribute keys between siblings | Rewrite separator in parent | None |

> **Rebalancing** (redistributing between neighbours without merging) is a common optimisation for reducing split/merge churn on workloads that hover around the threshold.

---

## Concrete Numbers: Fanout vs Height vs Seeks

The chapter's headline intuition — "high fanout means tiny height" — is the kind of thing best felt in numbers. The cell below compares BST-style fanout-2 trees with B-Trees of realistic fanouts (100, 200, 500) over a 4-billion-key index, and reports levels, disk seeks per lookup, and in-memory comparisons per lookup.

In [ ]:
# Stdlib-only, Python 3.11+
import math

M = 4_000_000_000  # 4 billion keys — the chapter's canonical example

fanouts = [2, 100, 200, 500]  # 2 = BST baseline

print(f"Indexing M = {M:,} keys\n")
print(f"{'Fanout K':>10}{'Height (levels)':>20}{'Seeks per lookup':>22}{'Comparisons (log2 M)':>26}")
print("-" * 78)

for K in fanouts:
    # Height = ceil(log_K M). For a B-Tree, root-to-leaf traversal visits 'height' pages,
    # so page transfers (seeks) equals height.
    height = math.ceil(math.log(M, K))

    # Key comparisons: binary search inside each visited node.
    # Summed across all levels, this is ~ log2 M regardless of K (to a constant factor).
    comparisons = math.ceil(math.log2(M))

    label = f"{K}" + ("  (BST)" if K == 2 else "")
    print(f"{label:>10}{height:>20}{height:>22}{comparisons:>26}")

# Concrete memory/latency feel: treat a seek as 10 ms (HDD) or 0.15 ms (SSD).
print("\nLookup latency upper bound, assuming cold cache (no page in buffer pool):")
print(f"{'Fanout K':>10}{'Height':>10}{'HDD (10 ms/seek)':>22}{'SSD (0.15 ms/seek)':>24}")
print("-" * 66)
for K in fanouts:
    height = math.ceil(math.log(M, K))
    hdd_ms = height * 10.0
    ssd_ms = height * 0.15
    label = f"{K}" + ("  (BST)" if K == 2 else "")
    print(f"{label:>10}{height:>10}{hdd_ms:>18.1f} ms{ssd_ms:>20.2f} ms")

# How much of a single 8 KB page does a node actually fill, for the chosen fanouts?
# Assume ~16 bytes per (separator key + child pointer) entry.
print("\nPage budgeting: does fanout K even fit in a typical 8 KB page?")
page_size = 8 * 1024
entry_bytes = 16  # separator key + child pointer, approx
print(f"{'Fanout K':>10}{'Entries * 16 B':>20}{'Fits in 8 KB page?':>24}")
print("-" * 54)
for K in fanouts:
    needed = K * entry_bytes
    fits = "yes" if needed <= page_size else "NO — use bigger page"
    print(f"{K:>10}{needed:>18} B{fits:>24}")

# The BST-vs-B-Tree punchline
print("\nPunchline:")
bst_height = math.ceil(math.log2(M))
btree_height_500 = math.ceil(math.log(M, 500))
ratio = bst_height / btree_height_500
print(f"  BST over 4B keys:           {bst_height} levels -> {bst_height} cold seeks")
print(f"  B-Tree fanout=500:          {btree_height_500} levels -> {btree_height_500} cold seeks")
print(f"  B-Tree is ~{ratio:.1f}x shallower -> ~{ratio:.1f}x fewer seeks per lookup")
print(f"  Comparisons stay at ~{math.ceil(math.log2(M))} for both — CPU work is similar,")
print(f"  but the B-Tree does those comparisons on RAM-resident pages, not per-seek.")

---

## Key Takeaways

1. **BSTs are a RAM data structure.** Binary fanout, random node placement, and rotation-heavy balancing all translate into disk pain. 4 billion keys in a BST is 32 levels and potentially 32 seeks per lookup.
2. **The disk is a block device.** Whether HDD (expensive seeks, sector transfers) or SSD (page reads, erase-block rewrites, FTL-managed GC), every access moves a whole block. Design the data structure to make that block earn its keep.
3. **High fanout + low height is the core idea.** Fanout and height are inversely related. 500-ary fanout brings 4 billion keys within 4 levels; 100-ary within 5.
4. **Two cost views of a B-Tree lookup.** Page transfers go as `log_K M` (disk seeks — what you pay in I/O). Comparisons go as `log2 M` (CPU work — cheap once the pages are in memory).
5. **B+ Trees push values to leaves.** This keeps internal nodes dense with separators, raises fanout, and makes range scans linear along a sibling-linked leaf chain.
6. **Growth is bottom-up.** New leaves appear when old ones split, and only splits that reach the root increase tree height. The structure is self-balancing by construction.
7. **Splits promote; merges demote.** Overflowing a node promotes a separator up; merging two underflowing children demotes the parent's separator into the merged node. Both can cascade, but only root-level splits/merges change height.
8. **Accept some slack.** Occupancy sits typically between 50% and ~70% so that future inserts don't immediately split. Rebalancing (redistributing between siblings) damps the split/merge churn.

## See Also

- [[01-bst-limitations-for-disk]]
- [[02-disk-hardware-and-block-io]]
- [[03-on-disk-structure-design-principles]]
- [[04-btree-hierarchy-and-separator-keys]]
- [[05-btree-lookup-algorithm-and-complexity]]
- [[06-btree-node-splits-and-merges]]